# Stage 19 Transition-Regularization Ablation

This notebook launches the validation-only Stage 19 ablation for the Stage 16 61-epoch frozen-embedding TCN ensemble. It estimates the transition penalty from training labels only, reuses the completed Stage 16 ensemble as `lambda_transition = 0.0`, and trains the nonzero lambda grid only when the guarded run flag is enabled. The held-out test split is not evaluated here.

In [1]:
from pathlib import Path
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.transition_regularization import (
    DEFAULT_STAGE16_REPLICATION_OUTPUT_DIR,
    DEFAULT_STAGE19_OUTPUT_DIR,
    STAGE19_LAMBDAS,
    STAGE19_SEEDS,
    run_stage19_transition_regularization,
)

stage16_replication_dir = repo_root / DEFAULT_STAGE16_REPLICATION_OUTPUT_DIR
stage19_output_dir = repo_root / DEFAULT_STAGE19_OUTPUT_DIR

{
    "repo_root": str(repo_root),
    "stage16_replication_dir_exists": stage16_replication_dir.exists(),
    "stage19_output_dir": str(stage19_output_dir),
    "lambdas": list(STAGE19_LAMBDAS),
    "seeds": list(STAGE19_SEEDS),
}


{'repo_root': '/home/manns79/dreamt-wearable-sleep-staging',
 'stage16_replication_dir_exists': True,
 'stage19_output_dir': '/home/manns79/dreamt-wearable-sleep-staging/results/stage19_transition_regularization',
 'lambdas': [0.001, 0.01, 0.05],
 'seeds': [42, 43, 44]}

## Run Guard

Set `RUN_STAGE19_TRANSITION_REGULARIZATION = True` to train the nine nonzero-lambda seed runs and build the per-lambda ensembles. Completed seed runs are reused when their summary and validation prediction artifacts already exist.

In [2]:
RUN_STAGE19_TRANSITION_REGULARIZATION = True

stage19_summary = pd.DataFrame()
if RUN_STAGE19_TRANSITION_REGULARIZATION:
    stage19_summary = run_stage19_transition_regularization(
        output_dir=stage19_output_dir,
        stage16_replication_dir=stage16_replication_dir,
        lambdas=STAGE19_LAMBDAS,
        seeds=STAGE19_SEEDS,
        skip_completed=True,
    )
    display(stage19_summary)
    print("outputs:", stage19_output_dir)
else:
    summary_path = stage19_output_dir / "experiment_summary.csv"
    if summary_path.exists():
        stage19_summary = pd.read_csv(summary_path)
        display(stage19_summary)
    else:
        print("Stage 19 transition-regularization run is configured but not run.")


,stage,stage19_run_status,model_family,lambda_transition,output_dir,model,split,accuracy,balanced_accuracy,macro_f1,...,ensemble_method,member_seeds,n_ensemble_members,validation_true_transition_count,validation_predicted_transition_count,true_wake_to_rem_transition_count,predicted_wake_to_rem_transition_count,true_wake_to_rem_transition_rate,predicted_wake_to_rem_transition_rate,rem_duration_error_status
0,stage19,reused_stage16_ensemble,stage16_equal_weight_seed_ensemble,0.000,/home/manns79/dreamt-wearable-sleep-staging/re...,stage16_equal_weight_seed_ensemble,validation,0.661345,0.515297,0.506251,...,equal_probability_average,"[42, 43, 44]",3,12011,12011,33,5,0.002747,0.000416,not_available
1,stage19,trained,transition_regularized_frozen_stage14_tcn_s61,0.001,/home/manns79/dreamt-wearable-sleep-staging/re...,stage19_transition_regularized_ensemble_lambda...,validation,0.660181,0.514985,0.505411,...,equal_probability_average,"[42, 43, 44]",3,12011,12011,33,6,0.002747,0.000500,not_available
2,stage19,trained,transition_regularized_frozen_stage14_tcn_s61,0.010,/home/manns79/dreamt-wearable-sleep-staging/re...,stage19_transition_regularized_ensemble_lambda...,validation,0.664422,0.513635,0.506201,...,equal_probability_average,"[42, 43, 44]",3,12011,12011,33,4,0.002747,0.000333,not_available
3,stage19,trained,transition_regularized_frozen_stage14_tcn_s61,0.050,/home/manns79/dreamt-wearable-sleep-staging/re...,stage19_transition_regularized_ensemble_lambda...,validation,0.663258,0.518628,0.509597,...,equal_probability_average,"[42, 43, 44]",3,12011,12011,33,3,0.002747,0.000250,not_available


outputs: /home/manns79/dreamt-wearable-sleep-staging/results/stage19_transition_regularization


## Baseline Comparison

After the guarded run completes, this cell displays the lambda-zero baseline and nonzero-lambda ensemble deltas saved by the Stage 19 helper.

In [3]:
comparison_path = stage19_output_dir / "baseline_comparison.csv"
if comparison_path.exists():
    comparison = pd.read_csv(comparison_path)
    display(
        comparison[
            [
                "lambda_transition",
                "stage19_run_status",
                "accuracy",
                "balanced_accuracy",
                "macro_f1",
                "Wake_f1",
                "Non_REM_f1",
                "REM_f1",
                "macro_f1_delta_vs_lambda_0",
                "predicted_wake_to_rem_transition_count",
                "true_wake_to_rem_transition_count",
            ]
        ]
    )
else:
    print("No Stage 19 baseline comparison has been written yet.")


,lambda_transition,stage19_run_status,accuracy,balanced_accuracy,macro_f1,Wake_f1,Non_REM_f1,REM_f1,macro_f1_delta_vs_lambda_0,predicted_wake_to_rem_transition_count,true_wake_to_rem_transition_count
0,0.000,reused_stage16_ensemble,0.661345,0.515297,0.506251,0.475620,0.771265,0.271870,0.000000,5,33
1,0.001,trained,0.660181,0.514985,0.505411,0.477182,0.770180,0.268872,-0.000840,6,33
2,0.010,trained,0.664422,0.513635,0.506201,0.476907,0.774458,0.267238,-0.000050,4,33
3,0.050,trained,0.663258,0.518628,0.509597,0.480967,0.772572,0.275253,0.003346,3,33
